# 波士顿房价预测：经典机器学习教程

[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/greathousesh/transformer-tutorial/blob/main/boston_housing_tutorial.ipynb)

本 notebook 通过 **Boston Housing**（波士顿房价）这一机器学习入门经典数据集，
完整演示一个**端到端的回归任务**：从数据加载、探索性分析（EDA）、特征工程，
到训练多种模型并进行对比、调优、可解释性分析。

## 目标读者
- 刚接触机器学习、想要走完一个完整流程的同学
- 想复习线性回归 / 正则化 / 树模型 / 集成学习的工程师
- 想了解残差分析、特征重要性等模型诊断手段的人

## 目录
1. 数据集介绍与加载
2. 探索性数据分析（EDA）
3. 特征关系与相关性可视化
4. 数据预处理（划分、标准化）
5. 基线模型：线性回归
6. 正则化：Ridge & Lasso
7. 树模型：决策树、随机森林、梯度提升
8. 交叉验证与超参搜索
9. 模型对比汇总
10. 特征重要性分析
11. 残差诊断
12. 深度学习对照：PyTorch MLP
13. 总结与延伸

> ⚠️ **关于数据集的说明**：sklearn 自 1.2 起已移除 `load_boston`，因为该数据集存在
> 伦理争议（一个特征隐含种族偏见）。本 notebook 仅用于**教学演示**机器学习流程，
> 在真实项目中请使用 `fetch_california_housing` 等替代数据集。


## 1. 环境与依赖

本教程仅依赖标准的 PyData + scikit-learn 工具链，Kaggle 默认环境已全部预装。


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

print(f'numpy   : {np.__version__}')
print(f'pandas  : {pd.__version__}')
import sklearn
print(f'sklearn : {sklearn.__version__}')


## 2. 数据集介绍与加载

**Boston Housing** 数据集包含波士顿市郊 506 条房屋记录，目标变量 `MEDV` 是该地区房价中位数（单位：千美元）。
13 个输入特征覆盖了人口、犯罪率、房屋结构、教育、可达性等维度：

| 特征 | 含义 |
| --- | --- |
| CRIM | 城镇人均犯罪率 |
| ZN | 占地超过 25,000 平方尺的住宅用地比例 |
| INDUS | 城镇非零售业商业用地比例 |
| CHAS | 是否邻近 Charles 河（1=邻近，0=不邻近） |
| NOX | 一氧化氮浓度 |
| RM | 每户平均房间数 |
| AGE | 自住单位中 1940 年前建成比例 |
| DIS | 到 5 个波士顿就业中心的加权距离 |
| RAD | 高速公路可达性指数 |
| TAX | 每万元财产税率 |
| PTRATIO | 师生比例 |
| B | 与镇内黑人人口比例相关的指标（含伦理争议） |
| LSTAT | 低收入人口比例 |
| **MEDV** | **房价中位数（目标）** |

由于 sklearn 已移除该数据集，我们直接从原始 CMU 镜像加载，并提供 OpenML 后备方案，
保证在 Kaggle 与本地都能正常运行。


In [ ]:
def load_boston_dataset():
    """从 CMU 原始镜像加载 Boston Housing；失败时回退到 OpenML。"""
    feature_names = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE',
                     'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']
    try:
        url = 'http://lib.stat.cmu.edu/datasets/boston'
        raw = pd.read_csv(url, sep=r'\s+', skiprows=22, header=None)
        data = np.hstack([raw.values[::2, :], raw.values[1::2, :2]])
        target = raw.values[1::2, 2]
        df = pd.DataFrame(data, columns=feature_names)
        df['MEDV'] = target
        print('已从 CMU 镜像加载数据')
        return df
    except Exception as e:
        print(f'CMU 镜像不可用 ({e})，回退至 OpenML...')
        from sklearn.datasets import fetch_openml
        boston = fetch_openml(name='boston', version=1, as_frame=True, parser='auto')
        df = boston.frame.copy()
        df.columns = [c.upper() for c in df.columns]
        df['CHAS'] = df['CHAS'].astype(float)
        df['RAD'] = df['RAD'].astype(float)
        df['MEDV'] = df['MEDV'].astype(float)
        return df[feature_names + ['MEDV']]


df = load_boston_dataset()
print(f'数据形状: {df.shape}')
df.head()


## 3. 探索性数据分析（EDA）

EDA 是建模前必做的一步：了解数据的形状、量纲、缺失情况、异常值与目标变量的分布。


In [ ]:
print('数据类型与非空数量:')
df.info()

print('\n缺失值统计:')
print(df.isnull().sum())

print('\n描述性统计:')
df.describe().round(2)


### 3.1 目标变量分布

观察 `MEDV` 的分布：是否对称？是否需要做 log 变换？是否有截断（封顶）？


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['MEDV'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('MEDV (House Price) Distribution')
axes[0].set_xlabel('MEDV (in $1000s)')
axes[0].set_ylabel('Count')

axes[1].boxplot(df['MEDV'], vert=False)
axes[1].set_title('MEDV Boxplot')
axes[1].set_xlabel('MEDV (in $1000s)')

plt.tight_layout()
plt.show()

print(f"MEDV 值为 50（封顶）的记录数: {(df['MEDV'] == 50).sum()}")
print(f"偏度 (skewness):  {df['MEDV'].skew():.3f}")
print(f"峰度 (kurtosis): {df['MEDV'].kurt():.3f}")


**观察**：
- `MEDV` 分布右偏（skewness > 0），尾部有少量高价房；
- 在 50 处出现明显堆积，说明数据被**截断在 \$50,000**。这是经典的右截断问题，
  会让模型对高价区域的预测产生偏差（这里我们保留以演示残差分析时的现象）。


## 4. 特征关系与相关性可视化

### 4.1 相关性热图

相关性热图能快速识别：
- 哪些特征与目标 `MEDV` 关联强（候选重要特征）；
- 哪些特征之间高度相关（共线性，影响线性模型解释）。


In [ ]:
corr = df.corr()

plt.figure(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=13)
plt.tight_layout()
plt.show()

print('与 MEDV 相关性最强的 5 个特征:')
print(corr['MEDV'].drop('MEDV').abs().sort_values(ascending=False).head(5))


**关键发现**：
- `LSTAT`（低收入人口比例）与 `MEDV` 强负相关（≈ -0.74）；
- `RM`（房间数）与 `MEDV` 强正相关（≈ +0.70）；
- `RAD` 与 `TAX` 高度正相关（≈ 0.91），是显著的多重共线性。


### 4.2 顶部相关特征 vs 房价的散点图

可视化最相关的几个特征 vs `MEDV`，帮助判断关系是线性还是非线性。


In [ ]:
top_features = ['LSTAT', 'RM', 'PTRATIO', 'INDUS', 'NOX', 'TAX']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, feat in zip(axes.flat, top_features):
    ax.scatter(df[feat], df['MEDV'], alpha=0.4, s=15, color='steelblue')
    ax.set_xlabel(feat)
    ax.set_ylabel('MEDV')
    ax.set_title(f'{feat} vs MEDV')

plt.tight_layout()
plt.show()


**观察**：
- `LSTAT` 与 `MEDV` 呈现明显**非线性**关系（曲线下降），暗示树模型或多项式特征会有优势；
- `RM` 大致线性，但在房间数 ≥ 8 时分布发散；
- `PTRATIO`、`INDUS` 等也展现非线性结构。


## 5. 数据预处理

### 5.1 训练 / 测试划分

- 80% 训练，20% 测试，固定随机种子保证可复现；
- 后续将在训练集上做 5 折交叉验证，避免泄漏测试集。


In [ ]:
X = df.drop(columns='MEDV').values
y = df['MEDV'].values
feature_names = df.drop(columns='MEDV').columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f'训练集: X={X_train.shape}, y={y_train.shape}')
print(f'测试集: X={X_test.shape},  y={y_test.shape}')


### 5.2 标准化

线性模型与正则化模型对量纲敏感（梯度收敛、L1/L2 惩罚都受量纲影响），
因此对线性族模型使用 `StandardScaler`；树模型对量纲不敏感，直接用原始数据。

> **重要**：`scaler.fit` 只在训练集上做，再 `transform` 测试集，避免数据泄漏。


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('标准化后训练集前 3 行:')
pd.DataFrame(X_train_scaled, columns=feature_names).head(3).round(3)


### 5.3 评估辅助函数

定义一个统一的评估器，输出 RMSE / MAE / R²，并支持 5 折交叉验证。


In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te, name):
    model.fit(X_tr, y_tr)
    pred_tr = model.predict(X_tr)
    pred_te = model.predict(X_te)

    cv = cross_val_score(model, X_tr, y_tr, cv=5,
                         scoring='neg_root_mean_squared_error')
    cv_rmse = -cv.mean()
    cv_std = cv.std()

    return {
        'model': name,
        'train_rmse': np.sqrt(mean_squared_error(y_tr, pred_tr)),
        'test_rmse': np.sqrt(mean_squared_error(y_te, pred_te)),
        'test_mae': mean_absolute_error(y_te, pred_te),
        'test_r2': r2_score(y_te, pred_te),
        'cv_rmse': cv_rmse,
        'cv_std': cv_std,
        'fitted': model,
    }


def print_result(r):
    print(f"{r['model']:<22} | "
          f"Train RMSE: {r['train_rmse']:.3f} | "
          f"Test RMSE: {r['test_rmse']:.3f} | "
          f"Test MAE: {r['test_mae']:.3f} | "
          f"Test R²: {r['test_r2']:.3f} | "
          f"CV RMSE: {r['cv_rmse']:.3f} ± {r['cv_std']:.3f}")


results = []


## 6. 基线模型：线性回归

线性回归 ($\hat y = w^\top x + b$) 是最经典的基线，便于解释，
也能让我们看到「不做任何花哨的事，模型能到什么水平」。


In [ ]:
r = evaluate(LinearRegression(), X_train_scaled, y_train,
             X_test_scaled, y_test, 'Linear Regression')
print_result(r)
results.append(r)

coef = pd.Series(r['fitted'].coef_, index=feature_names).sort_values()
plt.figure(figsize=(8, 5))
coef.plot(kind='barh', color=['#d62728' if v < 0 else '#2ca02c' for v in coef])
plt.title('Linear Regression Coefficients (on standardized features)')
plt.xlabel('Coefficient')
plt.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()


**解读系数**（特征已标准化，可直接比较绝对值大小）：
- `LSTAT` 系数最负 → 低收入比例越高，房价越低；
- `RM` 系数最正 → 房间越多，房价越高；
- `DIS` 为负，与直觉相反——这是与 `RAD` / `TAX` 共线性导致的「系数翻转」，
  在线性模型中常见，下面正则化模型会缓解。


## 7. 正则化：Ridge 与 Lasso

- **Ridge (L2)**: $\min \|y - Xw\|^2 + \alpha \|w\|_2^2$，缩小所有系数；
- **Lasso (L1)**: $\min \|y - Xw\|^2 + \alpha \|w\|_1$，会把不重要特征的系数压到 0（自动特征选择）。

我们用 `GridSearchCV` 在训练集上选出最佳 $\alpha$。


In [ ]:
ridge_grid = GridSearchCV(
    Ridge(),
    param_grid={'alpha': np.logspace(-3, 3, 25)},
    cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
ridge_grid.fit(X_train_scaled, y_train)
print(f'Ridge 最佳 alpha: {ridge_grid.best_params_["alpha"]:.4f}')

lasso_grid = GridSearchCV(
    Lasso(max_iter=20000),
    param_grid={'alpha': np.logspace(-3, 1, 25)},
    cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
lasso_grid.fit(X_train_scaled, y_train)
print(f'Lasso 最佳 alpha: {lasso_grid.best_params_["alpha"]:.4f}')

r_ridge = evaluate(Ridge(alpha=ridge_grid.best_params_['alpha']),
                   X_train_scaled, y_train, X_test_scaled, y_test, 'Ridge')
r_lasso = evaluate(Lasso(alpha=lasso_grid.best_params_['alpha'], max_iter=20000),
                   X_train_scaled, y_train, X_test_scaled, y_test, 'Lasso')
print_result(r_ridge)
print_result(r_lasso)
results.extend([r_ridge, r_lasso])


In [ ]:
coef_df = pd.DataFrame({
    'Linear': results[0]['fitted'].coef_,
    'Ridge':  r_ridge['fitted'].coef_,
    'Lasso':  r_lasso['fitted'].coef_,
}, index=feature_names).sort_values('Linear')

coef_df.plot(kind='barh', figsize=(9, 6), width=0.8)
plt.title('Coefficient Comparison: Linear vs Ridge vs Lasso')
plt.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

zeroed = coef_df.index[(coef_df['Lasso'] == 0)].tolist()
print(f'Lasso 压到 0 的特征: {zeroed}')


**观察**：
- Ridge 把所有系数整体收缩，缓解共线性；
- Lasso 把若干特征系数压到 0，做了**自动特征选择**；
- 在共线特征对（如 `RAD` 与 `TAX`）中，Lasso 倾向于保留一个、丢弃另一个。


## 8. 树模型：决策树、随机森林、梯度提升

树模型直接处理原始特征，能自然捕捉非线性、特征间的交互，且不需要标准化。
我们使用**未标准化**的 `X_train` / `X_test`。


In [ ]:
r_dt = evaluate(DecisionTreeRegressor(max_depth=6, random_state=42),
                X_train, y_train, X_test, y_test, 'Decision Tree (d=6)')

r_rf = evaluate(RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
                X_train, y_train, X_test, y_test, 'Random Forest')

r_gb = evaluate(GradientBoostingRegressor(n_estimators=300,
                                          learning_rate=0.05,
                                          max_depth=3,
                                          random_state=42),
                X_train, y_train, X_test, y_test, 'Gradient Boosting')

for r in [r_dt, r_rf, r_gb]:
    print_result(r)
results.extend([r_dt, r_rf, r_gb])


**观察**：
- 单棵决策树容易过拟合（训练 RMSE 远低于测试 RMSE）；
- 随机森林通过 bagging 显著降低方差，泛化更稳；
- 梯度提升通常拿到最好的成绩——它逐棵拟合上一轮的残差，是 Kaggle 表格类竞赛的常胜将军。


### 8.1 对梯度提升做超参搜索

简单网格搜索演示，覆盖学习率 × 深度 × 树数：


In [ ]:
gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid={
        'n_estimators': [200, 400],
        'learning_rate': [0.03, 0.05, 0.1],
        'max_depth': [2, 3, 4],
    },
    cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1)
gb_grid.fit(X_train, y_train)

print(f'GBR 最佳参数: {gb_grid.best_params_}')
print(f'GBR 最佳 CV RMSE: {-gb_grid.best_score_:.3f}')

r_gb_tuned = evaluate(GradientBoostingRegressor(**gb_grid.best_params_, random_state=42),
                      X_train, y_train, X_test, y_test, 'GBR (tuned)')
print_result(r_gb_tuned)
results.append(r_gb_tuned)


## 9. 模型对比汇总

把所有模型按测试集 RMSE 排序，一眼看清谁更强。


In [ ]:
leaderboard = pd.DataFrame([
    {'Model': r['model'],
     'Train RMSE': r['train_rmse'],
     'Test RMSE':  r['test_rmse'],
     'Test MAE':   r['test_mae'],
     'Test R²':    r['test_r2'],
     'CV RMSE':    r['cv_rmse']}
    for r in results
]).sort_values('Test RMSE').reset_index(drop=True)

leaderboard.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(leaderboard))
ax.bar(x - 0.2, leaderboard['Train RMSE'], 0.4, label='Train RMSE', color='lightsteelblue')
ax.bar(x + 0.2, leaderboard['Test RMSE'],  0.4, label='Test RMSE',  color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels(leaderboard['Model'], rotation=30, ha='right')
ax.set_ylabel('RMSE')
ax.set_title('Train vs Test RMSE per Model')
ax.legend()
plt.tight_layout()
plt.show()


**结论**：
- 集成树模型（RF、GBR）在 Test RMSE 上明显优于线性族；
- 注意**训练 RMSE 远小于测试 RMSE 通常意味着过拟合**——单棵决策树最为典型；
- 调过参的 GBR 在 CV 与 Test 上都最稳。


## 10. 特征重要性

树模型自带 `feature_importances_`，可以告诉我们模型最依赖哪些特征。


In [ ]:
best_tree_model = r_gb_tuned['fitted']
imp = pd.Series(best_tree_model.feature_importances_,
                index=feature_names).sort_values()

plt.figure(figsize=(8, 5))
imp.plot(kind='barh', color='seagreen')
plt.title('Feature Importance — Tuned Gradient Boosting')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print('Top 5 重要特征:')
print(imp.sort_values(ascending=False).head(5))


**与 EDA 的相关性结论一致**：`LSTAT` 与 `RM` 是最重要的两个特征。
特征重要性给出的是**预测贡献**，与相关性的「线性关联」是不同视角，但在本数据上结论吻合。


## 11. 残差诊断

残差 $r_i = y_i - \hat y_i$ 的分布与结构能告诉我们模型在哪些区域系统性偏差最大。
理想情况下，残差应该：
- 关于 0 对称；
- 与预测值无明显结构（无漏斗、无曲线）。


In [ ]:
best = leaderboard.iloc[0]['Model']
best_result = next(r for r in results if r['model'] == best)
best_model = best_result['fitted']

if best in ('Linear Regression', 'Ridge', 'Lasso'):
    pred = best_model.predict(X_test_scaled)
else:
    pred = best_model.predict(X_test)
residuals = y_test - pred

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(pred, y_test, alpha=0.6, edgecolor='white', s=30)
lim = [min(pred.min(), y_test.min()) - 2, max(pred.max(), y_test.max()) + 2]
axes[0].plot(lim, lim, 'r--', label='Perfect prediction')
axes[0].set_xlabel('Predicted MEDV')
axes[0].set_ylabel('True MEDV')
axes[0].set_title(f'Predicted vs True ({best})')
axes[0].legend()

axes[1].scatter(pred, residuals, alpha=0.6, edgecolor='white', s=30)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted MEDV')
axes[1].set_ylabel('Residual (y - ŷ)')
axes[1].set_title('Residual vs Predicted')

axes[2].hist(residuals, bins=30, color='steelblue', edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--')
axes[2].set_xlabel('Residual')
axes[2].set_ylabel('Count')
axes[2].set_title('Residual Distribution')

plt.tight_layout()
plt.show()

print(f'残差均值:    {residuals.mean():+.3f}')
print(f'残差标准差: {residuals.std():.3f}')


**典型现象**：
- 在房价 ≈ 50（被截断的上限）附近，模型一致**低估**真实值——这是右截断造成的固有偏差；
- 中段（20–30）残差最对称，模型在这一区段表现最好；
- 总体残差近似零均值，说明没有显著的系统性偏移。

**改进方向**（留给读者）：
- 对 `MEDV` 做 `log1p` 变换，缓解右偏；
- 排除 `MEDV == 50` 的截断样本，再训练一遍；
- 尝试 `XGBoost`、`LightGBM`、`CatBoost` 等更强的提升模型；
- 加入多项式特征 / 交互特征，给线性模型「升级」。


## 12. 深度学习对照：PyTorch MLP

到这里我们已经看过线性族、单棵树、bagging、boosting。一个自然的疑问：
**用神经网络会不会更好？** 这一节我们用 PyTorch 写一个简单的 MLP，
和已经调好的 GBR 同台对比。

> 💡 **预告**：在 506 行的小表格数据上，**精心调过的梯度提升基本不可被神经网络超越**——
> 这是 Kaggle 表格类竞赛多年来的共识。这一节的目的不是「赢」，而是：
> 1. 演示一个端到端的 PyTorch 训练流程；
> 2. 亲眼看见「数据规模 → 模型选择」的工程直觉；
> 3. 学会用早停（early stopping）+ Dropout 避免小数据上的过拟合。


### 12.1 设计思路

| 组件 | 选择 | 原因 |
| --- | --- | --- |
| 输入 | 标准化后的 13 维特征 | 神经网络对量纲极敏感 |
| 隐藏层 | `13 → 64 → 32 → 1`，ReLU | 两层非线性足够覆盖本数据 |
| 正则 | Dropout(p=0.2) + 早停 | 506 行数据极易过拟合 |
| 损失 | MSE | 与其它模型一致 |
| 优化器 | Adam(lr=1e-3) | 默认稳妥选项 |
| 验证 | 从训练集再切 15% 做验证 | 严格不碰测试集 |

> 注意我们使用**标准化后的 `X_train_scaled` / `X_test_scaled`**——和线性模型同款,
> 树模型的非标准化数据不适合喂给神经网络。


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch 版本: {torch.__version__}')
print(f'使用设备: {DEVICE}')


class MLPRegressor(nn.Module):
    def __init__(self, in_dim=13, hidden=(64, 32), p_drop=0.2):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(p_drop)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


model = MLPRegressor().to(DEVICE)
print(model)
print(f'参数量: {sum(p.numel() for p in model.parameters()):,}')


### 12.2 划分验证集 + DataLoader

我们再从训练集中切出 15% 作验证集（用于早停判定），测试集**继续封存**直到最终评估。


In [ ]:
from sklearn.model_selection import train_test_split as tts

X_tr, X_val, y_tr, y_val = tts(X_train_scaled, y_train,
                               test_size=0.15, random_state=42)

def to_tensor(arr, dtype=torch.float32):
    return torch.tensor(arr, dtype=dtype)

train_ds = TensorDataset(to_tensor(X_tr), to_tensor(y_tr))
val_ds   = TensorDataset(to_tensor(X_val), to_tensor(y_val))
test_ds  = TensorDataset(to_tensor(X_test_scaled), to_tensor(y_test))

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64)
test_loader  = DataLoader(test_ds, batch_size=64)

print(f'训练: {len(train_ds)}, 验证: {len(val_ds)}, 测试: {len(test_ds)}')


### 12.3 训练循环 + 早停

每个 epoch:
1. 在训练集上做完整一遍梯度下降；
2. 在验证集上算 RMSE;
3. 若验证 RMSE 在 `patience` 个 epoch 内没改进，则提前终止并恢复最佳权重。


In [ ]:
def train_mlp(model, train_loader, val_loader,
              epochs=300, lr=1e-3, weight_decay=1e-4, patience=25):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    history = {'train_rmse': [], 'val_rmse': []}
    best_val = float('inf')
    best_state = None
    epochs_no_improve = 0

    for ep in range(1, epochs + 1):
        model.train()
        sum_sq, n = 0.0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
            sum_sq += ((pred.detach() - yb) ** 2).sum().item()
            n += yb.numel()
        train_rmse = (sum_sq / n) ** 0.5

        model.eval()
        with torch.no_grad():
            sum_sq, n = 0.0, 0
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb)
                sum_sq += ((pred - yb) ** 2).sum().item()
                n += yb.numel()
            val_rmse = (sum_sq / n) ** 0.5

        history['train_rmse'].append(train_rmse)
        history['val_rmse'].append(val_rmse)

        if val_rmse < best_val - 1e-4:
            best_val = val_rmse
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if ep % 20 == 0 or ep == 1:
            print(f'epoch {ep:3d} | train RMSE {train_rmse:.3f} | val RMSE {val_rmse:.3f}'
                  f' | best val {best_val:.3f}')

        if epochs_no_improve >= patience:
            print(f'早停于 epoch {ep}（连续 {patience} 轮无改进）')
            break

    model.load_state_dict(best_state)
    return history, best_val


model = MLPRegressor().to(DEVICE)
history, best_val_rmse = train_mlp(model, train_loader, val_loader)
print(f'\n最佳验证 RMSE: {best_val_rmse:.3f}')


### 12.4 学习曲线

观察训练 / 验证 RMSE 随 epoch 变化,可以判断是否过拟合或欠拟合。


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history['train_rmse'], label='Train RMSE', color='steelblue')
plt.plot(history['val_rmse'], label='Val RMSE', color='darkorange')
plt.xlabel('Epoch')
plt.ylabel('RMSE')
plt.title('MLP Training Curve')
plt.legend()
plt.tight_layout()
plt.show()


### 12.5 测试集评估 + 加入对比

用早停后恢复的最佳权重在**封存的测试集**上做最终评估,然后追加到 leaderboard。


In [ ]:
model.eval()
with torch.no_grad():
    pred_train = model(to_tensor(X_train_scaled).to(DEVICE)).cpu().numpy()
    pred_test  = model(to_tensor(X_test_scaled).to(DEVICE)).cpu().numpy()

train_rmse = np.sqrt(mean_squared_error(y_train, pred_train))
test_rmse  = np.sqrt(mean_squared_error(y_test, pred_test))
test_mae   = mean_absolute_error(y_test, pred_test)
test_r2    = r2_score(y_test, pred_test)

mlp_result = {
    'model': 'PyTorch MLP',
    'train_rmse': train_rmse,
    'test_rmse': test_rmse,
    'test_mae': test_mae,
    'test_r2': test_r2,
    'cv_rmse': best_val_rmse,
    'cv_std': float('nan'),
    'fitted': model,
}
results.append(mlp_result)
print_result(mlp_result)


In [ ]:
leaderboard = pd.DataFrame([
    {'Model': r['model'],
     'Train RMSE': r['train_rmse'],
     'Test RMSE':  r['test_rmse'],
     'Test MAE':   r['test_mae'],
     'Test R²':    r['test_r2']}
    for r in results
]).sort_values('Test RMSE').reset_index(drop=True)

leaderboard.round(3)


### 12.6 讨论：为什么 MLP 不一定能赢?

在本数据上你大概率会观察到:**调过的梯度提升 ≥ MLP**。常见原因:

1. **数据量太小**:506 行远不够让神经网络发挥参数效率优势;
2. **特征已是结构化的「语义列」**:不像图像/文本那样需要表示学习,GBR 直接处理足够;
3. **GBR 自带正则**:学习率 + 树深 + 子采样三件套等价于强力正则,小数据上不易过拟合;
4. **MLP 的 inductive bias 不匹配表格数据**:树模型在「分段常数 + 特征交互」上更高效。

**什么时候应该上 DL?**
- 数据量到达万级以上;
- 输入是图像 / 文本 / 音频 / 时序等需要表示学习的领域;
- 表格数据特别大且需要在线推理(此时 TabNet / FT-Transformer 等专用架构才有竞争力)。

> 这条工程直觉对你今后选模型非常重要:**先用 GBR 起一条强基线,再判断是否需要 DL**,
> 而不是反过来。


## 13. 总结

本 notebook 走完了一个**完整的回归任务流程**：

| 步骤 | 关键点 |
| --- | --- |
| 数据加载 | 处理 sklearn 移除的兼容问题，提供后备方案 |
| EDA | 看分布、看缺失、看异常（截断） |
| 相关性 | 识别强特征 + 共线性 |
| 预处理 | 仅在训练集 fit scaler，避免泄漏 |
| 基线 | 线性回归是不可省略的对照 |
| 正则化 | Ridge 抗共线性，Lasso 自动选特征 |
| 树模型 | 不需标准化、自然捕捉非线性 |
| 调参 | GridSearchCV + 5 折 CV |
| 解释 | 系数 / 特征重要性 / 残差三件套 |
| 深度学习 | PyTorch MLP + 早停，作为对照而非首选 |

### 学到的核心思想
1. **永远先有一条基线**——没有基线，所有「提升」都没有意义；
2. **标准化只在线性族 / 神经网络上需要**，树模型不必；
3. **超参搜索必须在训练集内做交叉验证**，绝不碰测试集；
4. **残差诊断 ≠ 评估指标**——它告诉你「错在哪里」，比单一 RMSE 更有价值；
5. **特征重要性 ≠ 因果**——它只反映模型的依赖，不是真实世界的因果关系；
6. **小表格数据上 GBR 通常 ≥ 神经网络**——先有 GBR 强基线，再考虑是否上 DL。

### 进一步阅读
- *An Introduction to Statistical Learning* (ISLR) — 第 3、6、8 章
- scikit-learn 官方文档：`Pipeline`, `ColumnTransformer`, `cross_validate`
- 用 SHAP 值做更严谨的可解释性分析
- 表格数据上的现代 DL 架构：TabNet、FT-Transformer、SAINT
